# LightGBM 단독 5-Fold OOF 생성

torch/TabTransformer와 완전히 분리된 노트북입니다. LightGBM만 단독으로 사용하므로
스레드 충돌 걱정 없이 정상 속도로 동작합니다.

이 노트북의 결과(`blend_cache/oof_lgbm.npy`, `blend_cache/oof_y.npy`)는
`1차_blend_compare.ipynb`에서 TabTransformer 결과와 비교/블렌딩하는 데 사용됩니다.

## 1. 라이브러리 및 설정

In [1]:
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5
CACHE_DIR = "blend_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

## 2. 데이터 전처리 (main.py와 동일)

In [2]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

df_le = df.copy()
for col in cat_cols:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col].astype(str))

y = df_le[TARGET_COL].values.astype(np.float32)
X_lgbm = df_le.drop(columns=[TARGET_COL])

print(f"전처리 완료: {X_lgbm.shape}")

전처리 완료: (256351, 67)


## 3. 5-Fold OOF 생성

In [3]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=1004)
oof_lgbm = np.zeros(len(y))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_lgbm, y), start=1):
    m_lgbm = LGBMClassifier(random_state=1004, verbose=-1)
    m_lgbm.fit(X_lgbm.iloc[tr_idx], y[tr_idx])
    oof_lgbm[val_idx] = m_lgbm.predict_proba(X_lgbm.iloc[val_idx])[:, 1]
    fold_auc = roc_auc_score(y[val_idx], oof_lgbm[val_idx])
    print(f"Fold {fold}/{N_SPLITS}  LightGBM AUC: {fold_auc:.5f}")

score_lgbm = roc_auc_score(y, oof_lgbm)
print(f"\nLightGBM 5-Fold 전체 OOF AUC: {score_lgbm:.5f}")

Fold 1/5  LightGBM AUC: 0.74344
Fold 2/5  LightGBM AUC: 0.73912
Fold 3/5  LightGBM AUC: 0.73744
Fold 4/5  LightGBM AUC: 0.73819
Fold 5/5  LightGBM AUC: 0.73832

LightGBM 5-Fold 전체 OOF AUC: 0.73925


## 4. 결과 저장 (blend_compare 노트북에서 사용)

In [4]:
np.save(f"{CACHE_DIR}/oof_lgbm.npy", oof_lgbm)
np.save(f"{CACHE_DIR}/oof_y.npy", y)
print("저장 완료: blend_cache/oof_lgbm.npy, blend_cache/oof_y.npy")

저장 완료: blend_cache/oof_lgbm.npy, blend_cache/oof_y.npy
